In [1]:
#Focus: payment schema, tracking, late payment detection
import pandas as pd
from datetime import datetime, timedelta
from sqlalchemy import select
from src.database import SessionLocal
from src.models import Payment, Tenant, Property


In [2]:
print(
    """
    Payment Schema:
    -id: unique identifier
    -tenant_id: FK to Tenant
    -property_id: FK to Property
    -amount: payment amount (Decimal)
    -currency: ZAR, USD, etc.
    -status: 'pending', 'paid', 'overdue', 'late'
    -due-date: when payment is due
    -paid_at: when actually paid
    -reference: invoice/check reference
    -created_at, updated_at: timestamps
    """
)


    Payment Schema:
    -id: unique identifier
    -tenant_id: FK to Tenant
    -property_id: FK to Property
    -amount: payment amount (Decimal)
    -currency: ZAR, USD, etc.
    -status: 'pending', 'paid', 'overdue', 'late'
    -due-date: when payment is due
    -paid_at: when actually paid
    -reference: invoice/check reference
    -created_at, updated_at: timestamps
    


In [3]:
def add_payment(tenant_id, property_id, amount, currency="ZAR", due_date= None, status="pending", paid_at=None, reference=None):
    """Add a new payment record"""
    payment = Payment(
        tenant_id = tenant_id,
        property_id = property_id,
        amount = amount,
        currency = currency,
        status = status,
        due_date = due_date,
        reference = reference,
    )

    with SessionLocal() as session:
        session.add(payment)
        session.commit()
        session.refresh(payment)

    return payment

def list_payment():
    """Get all payments"""
    with SessionLocal() as session:
        return session.scalar(select(Payment)).all

def get_payment_by_id(payment_id):
    """Get payment by id"""
    with SessionLocal() as session:
        return session.get(Payment, payment_id)

def get_payments_by_tenant(tenant_id):
    """Get all payments for a tenant"""
    with SessionLocal() as session:
        return session.scalars(select(Payment).where(Payment.tenant_id==tenant_id)).all()

In [4]:
from datetime import timezone
def is_late(payment):
    """Check if payment is late"""
    if payment.status == "paid":
        return False

    if not payment.due_date:
        return False

    return datetime.now(payment.due_date.tzinfo) > payment.due_date

def get_overdue_payments():
    """Get all overdue payments"""
    with SessionLocal() as session:
        payments = session.scalars(select(Payment).where(Payment.status.in_(["pending","overdue"]))).all()
        overdue = [p for p in payments if is_late(p)]
    return overdue

def mark_payment_as_paid(payment_id):
    """Mark payment as paid"""
    with SessionLocal() as session:
        payment = session.get(Payment, payment_id)
        if payment:
            payment.status = "paid"
            payment.paid_at = datetime.now(timezone.utc)
            session.commit()
            session.refresh(payment)

    return payment

def update_payment_status(payment_id, status):
    """Update payment status"""
    with SessionLocal() as session:
        payment = session.get(Payment, payment_id)
        if payment:
            payment.status = status
            session.commit()
            session.refresh(payment)
    return payment

In [5]:
def get_tenant_payment_summer(tenant_id):
    """Get tenant payment summer"""
    payments = get_payments_by_tenant(tenant_id)

    total_owed = sum(p.amount for p in payments if p.status != "paid")
    total_paid = sum(p.amount for p in payments if p.status == "paid")
    overdue_count = sum(1 for p in payments if is_late(p))

    return {
        "tenant_id": tenant_id,
        "total_owed": total_owed,
        "total_paid": total_paid,
        "overdue_count": overdue_count,
        "payment_count": len(payments),
    }

In [6]:
from src.models import Tenant, Property
from src.database import Base, engine
Base.metadata.create_all(engine)

with SessionLocal() as session:
    tenant = session.get(Tenant, 1)

    if not tenant:
        print("Creating test tenant...")
        session.add_all([
            Tenant(id=1, full_name="Bennet Dyani", email="bennet@gmail.com", phone="+27630632730", unit_number="A1"),
            Tenant(id=2, full_name="Itumeleng Bedesho", email="itumeleng@gmail.com", phone="+27678238210", unit_number="B2"),

        ])
        session.commit()

    property_obj = session.get(Property, 1)
    if not property_obj:
        print("Creating test property...")
        session.add_all([
            Property(id=1, name="Sunrise Apartments", address="12 Main St", city="Cape Town", country="ZA"),
            Property(id=2, name="Green Valley Estate", address="45 Park Ave", city="Johannesburg", country="ZA"),
        ])
        session.commit()

payment1 = add_payment(
    tenant_id=1,
    property_id=1,
    amount=4500.00,
    currency="ZAR",
    due_date=datetime.now() - timedelta(days=10),
    status="pending",
    reference="INV-002",
)

payment2 = add_payment(
    tenant_id=2,
    property_id=2,
    amount=4500.00,
    currency="ZAR",
    due_date=datetime.now() - timedelta(days=10),
    status="pending",
    reference="INV-001",
)
print("Sample payment added:")
print(f"payment1 overdue: {payment1.amount} - Status: {payment1.status}")
print(f"payment2 pending: {payment2.amount} - Status: {payment2.status}")

print(f"\n====== Late Payments Detetction ======")
print(f"Payment 1 is late: {is_late(payment1)}")
print(f"Payment 2 is late: {is_late(payment2)}")

print(f"\n====== Tenant Payments Ssummary ======")
summary = get_tenant_payment_summer(1)
print(f"Total Owed: {summary['total_owed']}")
print(f"Total Paid: {summary['total_paid']}")
print(f"Overdue Count: {summary['overdue_count']}")

mark_payment_as_paid(payment1.id)
print(f"\nPayment 2 marked as paid")

Creating test property...
Sample payment added:
payment1 overdue: 4500.00 - Status: pending
payment2 pending: 4500.00 - Status: pending

====== Late Payments Detetction ======
Payment 1 is late: True
Payment 2 is late: True

====== Tenant Payments Ssummary ======
Total Owed: 4500.00
Total Paid: 0
Overdue Count: 1

Payment 2 marked as paid
